# Supervised attack classification: reservoir and classical head scaled together

Same data, protocol and reservoirs as `10_supervised_multiclass`; the head now
grows **with** the reservoir: each ladder step adds a bigger reservoir and a
wider MLP (config: `configs/supervised_ladder.yaml`). The original linear head
is trained at every step too (`lbfgs`, role *baseline*), so one run shows the
plateau and what the bigger head does to it.

Every head is trained on every arm at its step. That is what keeps the
comparison honest: an MLP on the `dials` learns its own nonlinearity, so the
reservoir only earns credit for beating `dials` **under the same head**; `rff`
asks whether the quantum lift beats a classical random lift of the same width.

In [ ]:
# Setup: make the repo importable from notebooks/ (or from the repo root).
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
sys.path.insert(0, str(REPO))

import pandas as pd
from elements import config
from elements.eval.results import load_results
from elements.experiments.runner import plan, run
from elements.viz import scaling
pd.set_option("display.precision", 4)
pd.set_option("display.width", 160)
from elements.viz import supervised as sv

In [ ]:
CFG = config.load(REPO / "configs" / "supervised_ladder.yaml")
FIGS = CFG.out_dir / "figs"
print(plan(CFG))

### Run

Off by default so opening the notebook never starts a long run. Set `RUN = True`
(or run `python -m elements.run configs/supervised_ladder.yaml` from the repo root). Every trained head is
appended to `results.jsonl` as it finishes, so an interrupted run resumes where
it stopped.

In [ ]:
RUN = False
if RUN:
    run(CFG)

## 1. The ladder

In [ ]:
DF = load_results(CFG.out_dir, keep=("per_class_f1", "confusion"))
scaling.ladder(DF, "macro_f1", out=FIGS / "ladder.png");

In [ ]:
LT = scaling.ladder_table(DF, "macro_f1")
LT

How to read it:

* **quantum solid vs quantum dashed** -- what the bigger head buys the reservoir.
* **quantum vs dials, both solid** -- the reservoir's own contribution once both
  sides have the same classical capacity. Under the linear head this gap is
  inflated, because a linear head cannot build nonlinearity from the dials.
* **quantum vs rff** -- quantum lift vs a classical random lift of equal width.
* **raw** -- the classical ceiling for that head size.

## 2. Where the scaled head changes things

In [ ]:
STEPS = scaling.ladder_steps(DF[DF.scheme == "all23"])
LAST = STEPS.iloc[-1]
sv.per_class_gain(DF, "all23", head=LAST["head"], out=FIGS / "per_class_gain_all23.png");

In [ ]:
sv.confusion(DF, "families6", LAST.config, LAST["head"], arms=("quantum", "dials", "rff"),
             out=FIGS / "confusion_last_step.png");

## 3. Operational view

In [ ]:
sv.operational_table(DF[DF.head_role == "scaled"])

## 4. Best setting per feature view

In [ ]:
scaling.best_table(DF, "macro_f1")

**Caveats.** One seed. MLP heads train on 90% of the train split (10% is
validation for early stopping); the linear baseline trains on the same 90% so
every head sees identical data. `all23` macro-F1 is dominated by classes with a
handful of rows -- quote `families6` for headline numbers.